# Code 2.9.3: Comparing hand-written and autograd gradients

Builds the two-moons MLP with `nn.Sequential`, copies in the initial weights of the from-scratch network, and compares the two sets of gradients on one batch of 64 examples. It uses `make_moons` from [`code/02-neural-network-basics/style.py`](style.py) and `init_params`, `forward`, `softmax_cross_entropy`, and `backward` from Section 2.7 (Code 2.7.2).

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.
# From Code 2.9.1
import torch

# Needed by make_moons
import numpy as np

# From Code 2.7.2
def init_params(n_in, n_hidden, n_out, seed=0):
    rng = np.random.default_rng(seed)
    return {
        "W1": rng.normal(0, np.sqrt(2 / n_in), (n_in, n_hidden)),
        "b1": np.zeros(n_hidden),
        "W2": rng.normal(0, np.sqrt(1 / n_hidden), (n_hidden, n_out)),
        "b2": np.zeros(n_out),
    }

def forward(P, X):
    Z1 = X @ P["W1"] + P["b1"]          # (B, H)   bias broadcasts over rows
    H = np.maximum(0, Z1)               # (B, H)
    Z2 = H @ P["W2"] + P["b2"]          # (B, C)   logits
    return Z2, (X, Z1, H)

def softmax_cross_entropy(Z, y):
    """Mean cross-entropy of integer labels y under softmax(Z); also returns dL/dZ."""
    Zs = Z - Z.max(axis=1, keepdims=True)        # stability: subtract the row max
    logsumexp = np.log(np.exp(Zs).sum(axis=1, keepdims=True))
    logp = Zs - logsumexp                        # log-softmax, (B, C)
    B = len(y)
    loss = -logp[np.arange(B), y].mean()
    dZ = np.exp(logp)                            # probabilities
    dZ[np.arange(B), y] -= 1.0                   # p - onehot(y)
    return loss, dZ / B

def backward(P, cache, dZ2):
    X, Z1, H = cache
    grads = {}
    grads["W2"] = H.T @ dZ2                      # (H, B) @ (B, C) -> (H, C)
    grads["b2"] = dZ2.sum(axis=0)                # (C,)
    dH = dZ2 @ P["W2"].T                         # (B, C) @ (C, H) -> (B, H)
    dZ1 = dH * (Z1 > 0)                          # ReLU passes gradient where Z1 > 0
    grads["W1"] = X.T @ dZ1                      # (D, B) @ (B, H) -> (D, H)
    grads["b1"] = dZ1.sum(axis=0)                # (H,)
    for k in P:
        assert grads[k].shape == P[k].shape, k   # every gradient matches its parameter
    return grads

# From code/02-neural-network-basics/style.py
def make_moons(n=200, noise=0.15, seed=0):
    """Two interleaving half-circles (our own implementation, no sklearn needed)."""
    rng = np.random.default_rng(seed)
    n0 = n // 2
    n1 = n - n0
    t0 = rng.uniform(0, np.pi, n0)
    t1 = rng.uniform(0, np.pi, n1)
    X0 = np.c_[np.cos(t0), np.sin(t0)]
    X1 = np.c_[1 - np.cos(t1), 0.5 - np.sin(t1)]
    X = np.r_[X0, X1] + rng.normal(0, noise, (n, 2))
    y = np.r_[np.zeros(n0, int), np.ones(n1, int)]
    perm = rng.permutation(n)
    return X[perm], y[perm]


In [ ]:
import numpy as np
import torch.nn as nn

X, y = make_moons(n=400, noise=0.2, seed=2)            # two-moons data from code/02-neural-network-basics/style.py
P0 = init_params(2, 16, 2, seed=0)                     # from-scratch initial weights (Section 2.7)

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
with torch.no_grad():                                  # copy weights without recording a graph
    model[0].weight.copy_(torch.from_numpy(P0["W1"].T))   # nn.Linear stores (out, in)
    model[0].bias.copy_(torch.from_numpy(P0["b1"]))
    model[2].weight.copy_(torch.from_numpy(P0["W2"].T))
    model[2].bias.copy_(torch.from_numpy(P0["b2"]))
loss_fn = nn.CrossEntropyLoss()
Xt, yt = torch.from_numpy(X), torch.from_numpy(y)

# Gradients on one batch of 64 examples: from scratch vs. autograd
Z, cache = forward(P0, X[:64])
_, dZ = softmax_cross_entropy(Z, y[:64])
ours = backward(P0, cache, dZ)
loss_fn(model(Xt[:64]), yt[:64]).backward()
theirs = {"W1": model[0].weight.grad.T, "b1": model[0].bias.grad,
          "W2": model[2].weight.grad.T, "b2": model[2].bias.grad}
for k in ours:
    diff = np.abs(ours[k] - theirs[k].numpy()).max()
    print(f"{k}: max |difference| = {diff:.1e}, allclose: {np.allclose(ours[k], theirs[k].numpy())}")


**Expected output**

Output:

```text
W2: max |difference| = 2.8e-17, allclose: True
b2: max |difference| = 2.3e-17, allclose: True
W1: max |difference| = 1.4e-17, allclose: True
b1: max |difference| = 1.4e-17, allclose: True
```
